In [1]:
%pip install pandas
%pip install wordninja
%pip install deep-translator
%pip install langdetect
%pip install nltk
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import re
import wordninja
import os
import string
import langdetect
import time
import nltk
nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('stopwords')
nltk.download('wordnet') 
from nltk.tokenize import word_tokenize, sent_tokenize
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from langdetect import detect
from deep_translator import GoogleTranslator


Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


[nltk_data] Downloading package punkt to C:\Users\Lish Ai
[nltk_data]     Labs\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to C:\Users\Lish Ai
[nltk_data]     Labs\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to C:\Users\Lish Ai
[nltk_data]     Labs\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to C:\Users\Lish Ai
[nltk_data]     Labs\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


In [2]:
#load dataset
file_path = 'C:/Users/Lish Ai Labs/Desktop/Simba/international_law_data.csv'
df = pd.read_csv(file_path)
print(df.head())


                             title  \
0                International law   
1   International humanitarian law   
2      Customary international law   
3  Condominium (international law)   
4       International criminal law   

                                             summary  \
0  International law, also known aspublic interna...   
1  \nInternational humanitarian law(IHL), also re...   
2  Customary international lawconsists ofinternat...   
3  \nAcondominium(plural eithercondominia, as in ...   
4  International criminal law(ICL) is a body ofpu...   

                                             content  \
0  International law, also known aspublic interna...   
1  \nNuremberg trials\nInternational Military Tri...   
2  Customary international lawconsists ofinternat...   
3  \nAcondominium(plural eithercondominia, as in ...   
4  International criminal law(ICL) is a body ofpu...   

                                               links  \
0  ['/wiki/Judge', '/wiki/Humphrey_Waldoc

In [3]:
print(df.dtypes)


title      object
summary    object
content    object
links      object
url        object
dtype: object


In [4]:
# checking for missing values
missing_values = df.isnull()
for column in missing_values.columns.values.tolist():
    print (missing_values[column].value_counts())
    print("")



title
False    9996
Name: count, dtype: int64

summary
False    8495
True     1501
Name: count, dtype: int64

content
False    8501
True     1495
Name: count, dtype: int64

links
False    9996
Name: count, dtype: int64

url
False    9996
Name: count, dtype: int64



In [5]:
#display the duplicate rows
duplicate_rows_df = df[df.duplicated()]
print("number of duplicate rows: ", duplicate_rows_df.shape)

number of duplicate rows:  (1167, 5)


In [6]:
#dropping duplicates
df = df.drop_duplicates()
print("number of duplicate rows: ", df.duplicated())

number of duplicate rows:  0       False
1       False
2       False
3       False
4       False
        ...  
9991    False
9992    False
9993    False
9994    False
9995    False
Length: 8829, dtype: bool


In [7]:
# Drop rows where both columns 'content' and 'summary' are empty
df1 = df.copy()
df_cleaned = df1.dropna(subset=['content', 'summary'], how='all')


In [8]:
#checking for missing values after removing duplicates
missing_valuesdf1 = df_cleaned.isnull()
for column in missing_valuesdf1.columns.values.tolist():
    print (missing_valuesdf1[column].value_counts())
    print("")

title
False    7471
Name: count, dtype: int64

summary
False    7466
True        5
Name: count, dtype: int64

content
False    7471
Name: count, dtype: int64

links
False    7471
Name: count, dtype: int64

url
False    7471
Name: count, dtype: int64



In [9]:
# having only 5 rows with missing we can replace them with Not Available

df1 = df.copy()

df_cleaned = df1.dropna(subset=['summary'], how='all').copy()

# Function to extract first 25 words
def get_summary(text):
    words = text.split()
    return ' '.join(words[:25]) if len(words) > 25 else text

df_cleaned.loc[df_cleaned['summary'].isna(), 'summary'] = df_cleaned.loc[df_cleaned['summary'].isna(), 'content'].apply(get_summary)


In [10]:
# checking if there are any missing values left
missing_valuesdf1 = df_cleaned.isnull()
for column in missing_valuesdf1.columns.values.tolist():
    print (missing_valuesdf1[column].value_counts())
    print("")

title
False    7466
Name: count, dtype: int64

summary
False    7466
Name: count, dtype: int64

content
False    7466
Name: count, dtype: int64

links
False    7466
Name: count, dtype: int64

url
False    7466
Name: count, dtype: int64



In [11]:
#removing \n from the content table and summary
df_cleaned["content"] = df_cleaned["content"].str.replace("\n", " ", regex=True)
df_cleaned["summary"] = df_cleaned["summary"].str.replace("\n", " ", regex=True)

In [ ]:
#checking for \n after replacing.
num_with_newline = df['content'].str.contains('\n').sum()
num_with_newline = df['summary'].str.contains('\n').sum()
num_with_newline = df['links'].str.contains('\n').sum()
num_with_newline = df['title'].str.contains('\n').sum()
num_with_newline = df['url'].str.contains('\n').sum()
print(f"Number of rows with newline characters: {num_with_newline}")


Number of rows with newline characters: 0


In [13]:
df_cleaned.head()


,title,summary,content,links,url
0,International law,"International law, also known aspublic interna...","International law, also known aspublic interna...","['/wiki/Judge', '/wiki/Humphrey_Waldock', '/wi...",https://en.wikipedia.org/wiki/International_law
1,International humanitarian law,"International humanitarian law(IHL), also ref...",Nuremberg trials International Military Tribu...,"['/wiki/Crime_of_aggression', '/wiki/Special:B...",https://en.wikipedia.org/wiki/International_hu...
2,Customary international law,Customary international lawconsists ofinternat...,Customary international lawconsists ofinternat...,"['/wiki/Judge', '/wiki/Legal_process_(jurispru...",https://en.wikipedia.org/wiki/Customary_intern...
3,Condominium (international law),"Acondominium(plural eithercondominia, as in L...","Acondominium(plural eithercondominia, as in L...","['/wiki/Archbishopric_of_Mainz', '/wiki/Benito...",https://en.wikipedia.org/wiki/Condominium_(int...
4,International criminal law,International criminal law(ICL) is a body ofpu...,International criminal law(ICL) is a body ofpu...,"['/wiki/Judge', '/wiki/Legal_intent', '/wiki/H...",https://en.wikipedia.org/wiki/International_cr...


In [14]:
# normalizing text
def normalize_text(text):
    if not isinstance(text, str):
        return ""
    text = text.lower()
    text = text.translate(str.maketrans('', '', string.punctuation))
    text = re.sub(r'\s+', ' ', text).strip()
    return text

def normalize_dataframe(df):
    for col in df.columns:
        if df[col].dtype == 'object':  
            try:
                df[col] = df[col].astype(str).apply(normalize_text) 
            except Exception as e:
                print(f"Error normalizing column {col}: {e}")
    return df

try:
    df = pd.read_csv(file_path)  

    # Normalize the entire DataFrame
    df = normalize_dataframe(df)

    # Print the first few rows to check the results
    print(df.head())

except FileNotFoundError:
    print("Error: CSV file not found.  Make sure the file path is correct.")
except Exception as e:
    print(f"An error occurred: {e}")

                            title  \
0               international law   
1  international humanitarian law   
2     customary international law   
3   condominium international law   
4      international criminal law   

                                             summary  \
0  international law also known aspublic internat...   
1  international humanitarian lawihl also referre...   
2  customary international lawconsists ofinternat...   
3  acondominiumplural eithercondominia as in lati...   
4  international criminal lawicl is a body ofpubl...   

                                             content  \
0  international law also known aspublic internat...   
1  nuremberg trials international military tribun...   
2  customary international lawconsists ofinternat...   
3  acondominiumplural eithercondominia as in lati...   
4  international criminal lawicl is a body ofpubl...   

                                               links  \
0  wikijudge wikihumphreywaldock wikilegalproce

In [15]:
# splitting words
df_cleaned = pd.read_csv(file_path)
print("DataFrame loaded successfully.") 
if 'summary' not in df_cleaned.columns or 'content' not in df_cleaned.columns:
            raise ValueError("The DataFrame must have 'summary' and 'content' columns.")
print("Columns 'summary' and 'content' verified.") 

df_cleaned['summary'] = df_cleaned['summary'].astype(str).apply(lambda x: " ".join(wordninja.split(x)))
print("Word splitting applied to 'summary' column.") 
df_cleaned['content'] = df_cleaned['content'].astype(str).apply(lambda x: " ".join(wordninja.split(x)))
print("Word splitting applied to 'content' column.") 

print("DataFrame processing completed successfully.") 


DataFrame loaded successfully.
Columns 'summary' and 'content' verified.
Word splitting applied to 'summary' column.
Word splitting applied to 'content' column.
DataFrame processing completed successfully.


In [16]:
df_cleaned['links'] = df['links']

df_cleaned.head()

,title,summary,content,links,url
0,International law,International law also known as public interna...,International law also known as public interna...,wikijudge wikihumphreywaldock wikilegalprocess...,https://en.wikipedia.org/wiki/International_law
1,International humanitarian law,International humanitarian law IHL also referr...,Nuremberg trials International Military Tribun...,wikicrimeofaggression wikispecialbooksources15...,https://en.wikipedia.org/wiki/International_hu...
2,Customary international law,Customary international law consists of intern...,Customary international law consists of intern...,wikijudge wikilegalprocessjurisprudence wikihi...,https://en.wikipedia.org/wiki/Customary_intern...
3,Condominium (international law),A condominium plural either condom in i a as i...,A condominium plural either condom in i a as i...,wikiarchbishopricofmainz wikibenitomussolini w...,https://en.wikipedia.org/wiki/Condominium_(int...
4,International criminal law,International criminal law ICL is a body of pu...,International criminal law ICL is a body of pu...,wikijudge wikilegalintent wikihistoryofthelega...,https://en.wikipedia.org/wiki/International_cr...


In [17]:
# removing links from the summary, content and title columns


def remove_url(text):
    return re.sub(r'https?://\S+|www\.\S+', '', text)

#This function removes punctuations
def remove_punct(text):
    return text.translate(str.maketrans('', '', string.punctuation))

df_cleaned['content'] = df_cleaned['content'].apply(lambda x: remove_url(x))
df_cleaned['summary'] = df_cleaned['summary'].apply(lambda x: remove_url(x))
df_cleaned['title'] = df_cleaned['title'].apply(lambda x: remove_url(x))

df_cleaned = df_cleaned.drop(columns=['links'])

In [18]:
# Define Cleaning Function (and Tokenization):
def clean_text(text):
    if not isinstance(text, str):
        return ""

    text = str(text).lower()
    tokens = word_tokenize(text)

    stop_words = set(stopwords.words('english'))
    tokens = [w for w in tokens if w not in stop_words and w.isalnum()]

    lemmatizer = WordNetLemmatizer() 
    tokens = [lemmatizer.lemmatize(w) for w in tokens]
    return tokens


text_columns = df.select_dtypes(include=['object']).columns
print(f"Identified text columns: {text_columns}")

for column in text_columns:
    df[column + '_cleaned'] = df[column].apply(clean_text)

print(df.head())

Identified text columns: Index(['title', 'summary', 'content', 'links', 'url'], dtype='object')
                            title  \
0               international law   
1  international humanitarian law   
2     customary international law   
3   condominium international law   
4      international criminal law   

                                             summary  \
0  international law also known aspublic internat...   
1  international humanitarian lawihl also referre...   
2  customary international lawconsists ofinternat...   
3  acondominiumplural eithercondominia as in lati...   
4  international criminal lawicl is a body ofpubl...   

                                             content  \
0  international law also known aspublic internat...   
1  nuremberg trials international military tribun...   
2  customary international lawconsists ofinternat...   
3  acondominiumplural eithercondominia as in lati...   
4  international criminal lawicl is a body ofpubl...   

       

In [19]:
# saving the cleaned dataset
 
folder_path = os.path.join(os.path.expanduser("~"), "Desktop", "Simba") 

file_name = "cleaned_international_law_data.csv"
file_path = os.path.join(folder_path, file_name)

df.to_csv(file_path, index=False, encoding='utf-8')

print(f"Cleaned data saved to: {file_path}")

Cleaned data saved to: C:\Users\Lish Ai Labs\Desktop\Simba\cleaned_international_law_data.csv
